In [ ]:
from langchain.agents import create_agent
#from langchain_groq import ChatGroq
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
from langchain.tools import tool
import requests

In [ ]:
load_dotenv()  # Load environment variables from .env file

In [ ]:
llm = ChatOllama(model="gpt-oss:120b-cloud")

In [ ]:
@tool
def get_weather(location="Pune", format_json=True):
    """
    Fetch weather data from wttr.in API.
    
    Args:
        location (str): City name or coordinates (e.g., "London", "48.8566,2.3522").
        format_json (bool): If True, returns JSON data; else returns plain text.
    
    Returns:
        dict | str: Weather data in JSON or text format.
    """
    try:
        if format_json:
            url = f"https://wttr.in/{location}?format=j1"  # JSON format
        else:
            url = f"https://wttr.in/{location}?format=3"   # Short text format
        
        response = requests.get(url, timeout=10)
        response.raise_for_status()  # Raise HTTPError for bad responses
        
        return response.json() if format_json else response.text
    
    except requests.exceptions.RequestException as e:
        return {"error": f"Failed to fetch weather data: {e}"}

In [ ]:
@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch latest stock price for a given symbol (e.g. 'AAPL', 'TSLA')
    using Alpha Vantage with API key in the URL.
    """
    url = f"https://www.alphavantage.co/query?function=GLOBAL_QUOTE&symbol={symbol}&apikey=7XV5E4IQS8ZF4ENY"
    r = requests.get(url)
    return r.json()

In [ ]:
# @tool
# def get_weather(city:str) -> str:
#     """
#     Simulates fetching weather information for a given city.
#     In a real-world scenario, this function would call an external API to get the weather data.
#     """
#     # For demonstration purposes, we'll return a mock response
#     return f"The current weather in {city} is sunny with a temperature of 28°C."

In [ ]:
get_weather.name

In [ ]:
get_weather.description

In [ ]:
get_weather.args

In [ ]:
get_weather

In [ ]:
my_agent = create_agent(
    model=llm,
    tools=[get_weather, get_stock_price],
    system_prompt="you are a helpful assistant that can provide weather and stock price information. Use the appropriate tools to fetch the required data when needed."
)

In [ ]:
my_agent

In [ ]:
response1 = my_agent.invoke({"messages":"Hello! Can you tell me the current weather in Pune City?"})

In [ ]:
response1["messages"][-1].content

In [ ]:
response2 = my_agent.invoke({"messages":"Hello! Can you tell me the current stock price of GOOGL?"})

In [ ]:
response2["messages"][-1].content